# Brain-to-Text '25 baseline inference (Kaggle-ready)
This notebook mirrors `model_training/evaluate_model.py` so it can run inside a Kaggle notebook. Update the paths in the configuration cell to point at your Kaggle input datasets (neural data, metadata CSV, and pretrained RNN). The pipeline loads the pretrained GRU decoder, performs inference for every trial in the chosen split, streams logits through the provided n-gram language model over Redis, and writes a `submission.csv` file to `/kaggle/working`.

In [ ]:
# Install any missing dependencies. Kaggle images include most of these,
# but the cell is left here for reproducibility.
!pip install -q omegaconf editdistance redis h5py

In [ ]:
import os
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import editdistance
from omegaconf import OmegaConf
from tqdm import tqdm

from model_training.rnn_model import GRUDecoder
from model_training.evaluate_model_helpers import (
    LOGIT_TO_PHONEME,
    rearrange_speech_logits_pt,
    runSingleDecodingStep,
    load_h5py_file,
    remove_punctuation,
    get_current_redis_time_ms,
    reset_remote_language_model,
    send_logits_to_remote_lm,
    finalize_remote_lm,
)
from model_training.evaluate_model_helpers import update_remote_lm_params  # optional

REPO_DIR = Path.cwd()
print(f'Working directory: {REPO_DIR}')

## Configure paths and runtime options
Edit the paths below to match the Kaggle datasets you attach to the notebook. If you mount this repository as a Kaggle Dataset, `REPO_DIR` will already point at it.

In [ ]:
# Locations of the pretrained RNN, neural data, and metadata CSV on Kaggle.
# Update these to match your dataset names.
DATA_DIR = Path('/kaggle/input/brain-to-text-25/hdf5_data_final')
CSV_PATH = Path('/kaggle/input/brain-to-text-25/t15_copyTaskData_description.csv')
MODEL_DIR = Path('/kaggle/input/brain-to-text-25/t15_pretrained_rnn_baseline')
# For Kaggle submissions, use the held-out test split. Use 'val' if you want to compute WER locally.
EVAL_SPLIT = 'test'  # or 'val'
# Set to -1 to force CPU inference. Kaggle GPU instances work with 0.
GPU_NUMBER = 0 if torch.cuda.is_available() else -1
# Path to the 1-gram LM shipped with the repo.
LM_PATH = REPO_DIR / 'language_model' / 'pretrained_language_models' / 'openwebtext_1gram_lm_sil'
# Redis host/port used by the language model process.
REDIS_HOST = 'localhost'
REDIS_PORT = 6379

# Where to write the submission file. Kaggle expects it in /kaggle/working.
SUBMISSION_PATH = Path('/kaggle/working/submission.csv')

# Sanity checks to make sure the paths exist.
for path in [DATA_DIR, CSV_PATH, MODEL_DIR, LM_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing required path: {path}')
print('Configured data directory:', DATA_DIR)
print('Configured model directory:', MODEL_DIR)

## Start Redis and the language model
The baseline decoding pipeline streams logits to the n-gram LM via Redis. If Redis is not installed in your Kaggle environment, uncomment the `apt-get` command below.

In [ ]:
# Ensure redis-server is available. Uncomment the install command if needed.
# !apt-get update && apt-get install -y redis-server

subprocess.run(['redis-server', '--version'], check=True)
subprocess.run(['redis-server', '--daemonize', 'yes', '--port', str(REDIS_PORT)], check=True)
print('Redis server started.')

In [ ]:
# Launch the lightweight 1-gram language model in the background.
lm_command = [
    'python', str(REPO_DIR / 'language_model' / 'language-model-standalone.py'),
    '--lm_path', str(LM_PATH),
    '--do_opt',
    '--nbest', '100',
    '--acoustic_scale', '0.325',
    '--blank_penalty', '90',
    '--alpha', '0.55',
    '--redis_ip', REDIS_HOST,
    '--gpu_number', str(max(0, GPU_NUMBER)),
]
lm_process = subprocess.Popen(lm_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print('Started language model process with PID', lm_process.pid)
time.sleep(5)  # give the LM a moment to initialize and connect to Redis

# Optionally, print a few lines of LM logs to confirm it started.
for _ in range(5):
    line = lm_process.stdout.readline().decode().strip()
    if line:
        print(line)

## Load the pretrained model and dataset
This mirrors `evaluate_model.py` but uses explicit variables instead of CLI flags so it works smoothly in Kaggle.

In [ ]:
# Load metadata
b2txt_csv_df = pd.read_csv(CSV_PATH)

# Load model hyperparameters
model_args = OmegaConf.load(MODEL_DIR / 'checkpoint' / 'args.yaml')

# Device selection
if torch.cuda.is_available() and GPU_NUMBER >= 0:
    if GPU_NUMBER >= torch.cuda.device_count():
        raise ValueError(f'GPU number {GPU_NUMBER} is out of range. Available GPUs: {torch.cuda.device_count()}')
    device = torch.device(f'cuda:{GPU_NUMBER}')
else:
    if GPU_NUMBER >= 0:
        print(f'GPU number {GPU_NUMBER} requested but not available, falling back to CPU.')
    device = torch.device('cpu')
print('Using device:', device)

# Build and load the GRU decoder
model = GRUDecoder(
    neural_dim=model_args['model']['n_input_features'],
    n_units=model_args['model']['n_units'],
    n_days=len(model_args['dataset']['sessions']),
    n_classes=model_args['dataset']['n_classes'],
    rnn_dropout=model_args['model']['rnn_dropout'],
    input_dropout=model_args['model']['input_network']['input_layer_dropout'],
    n_layers=model_args['model']['n_layers'],
    patch_size=model_args['model']['patch_size'],
    patch_stride=model_args['model']['patch_stride'],
)
checkpoint = torch.load(MODEL_DIR / 'checkpoint' / 'best_checkpoint', weights_only=False)
for key in list(checkpoint['model_state_dict'].keys()):
    checkpoint['model_state_dict'][key.replace('module.', '')] = checkpoint['model_state_dict'].pop(key)
    checkpoint['model_state_dict'][key.replace('_orig_mod.', '')] = checkpoint['model_state_dict'].pop(key)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()
print('Loaded pretrained checkpoint.')

# Load neural data for each session in the split
test_data = {}
total_trials = 0
for session in model_args['dataset']['sessions']:
    candidate = DATA_DIR / session / f'data_{EVAL_SPLIT}.hdf5'
    if candidate.exists():
        data = load_h5py_file(str(candidate), b2txt_csv_df)
        test_data[session] = data
        total_trials += len(data['neural_features'])
        print(f'Loaded {len(data["neural_features"])} {EVAL_SPLIT} trials for session {session}.')

print('Total number of trials:', total_trials)

## Run the neural decoder to obtain phoneme logits
Logits are computed in batches of one trial to mirror the original script.

In [ ]:
with tqdm(total=total_trials, desc='Predicting phoneme sequences', unit='trial') as pbar:
    for session, data in test_data.items():
        data['logits'] = []
        input_layer = model_args['dataset']['sessions'].index(session)
        for trial in range(len(data['neural_features'])):
            neural_input = np.expand_dims(data['neural_features'][trial], axis=0)
            dtype = torch.bfloat16 if device.type == 'cuda' else torch.float32
            neural_input = torch.tensor(neural_input, device=device, dtype=dtype)
            logits = runSingleDecodingStep(neural_input, input_layer, model, model_args, device)
            data['logits'].append(logits)
            pbar.update(1)
print('Finished decoding.')

## Stream logits through the language model and build predictions
This matches the Redis-based interface used in `evaluate_model.py`.

In [ ]:
import redis

r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, db=0)
r.flushall()

remote_lm_input_stream = 'remote_lm_input'
remote_lm_output_partial_stream = 'remote_lm_output_partial'
remote_lm_output_final_stream = 'remote_lm_output_final'

remote_lm_output_partial_lastEntrySeen = get_current_redis_time_ms(r)
remote_lm_output_final_lastEntrySeen = get_current_redis_time_ms(r)
remote_lm_done_resetting_lastEntrySeen = get_current_redis_time_ms(r)
remote_lm_done_finalizing_lastEntrySeen = get_current_redis_time_ms(r)
remote_lm_done_updating_lastEntrySeen = get_current_redis_time_ms(r)

lm_results = {
    'session': [],
    'block': [],
    'trial': [],
    'true_sentence': [],
    'pred_sentence': [],
}

with tqdm(total=total_trials, desc='Running language model', unit='trial') as pbar:
    for session in test_data.keys():
        for trial in range(len(test_data[session]['logits'])):
            logits = rearrange_speech_logits_pt(test_data[session]['logits'][trial])[0]
            remote_lm_done_resetting_lastEntrySeen = reset_remote_language_model(r, remote_lm_done_resetting_lastEntrySeen)
            # remote_lm_done_updating_lastEntrySeen = update_remote_lm_params(r, remote_lm_done_updating_lastEntrySeen)  # optional
            remote_lm_output_partial_lastEntrySeen, decoded = send_logits_to_remote_lm(
                r,
                remote_lm_input_stream,
                remote_lm_output_partial_stream,
                remote_lm_output_partial_lastEntrySeen,
                logits,
            )
            remote_lm_output_final_lastEntrySeen, lm_out = finalize_remote_lm(
                r,
                remote_lm_output_final_stream,
                remote_lm_output_final_lastEntrySeen,
            )
            best_sentence = lm_out['candidate_sentences'][0]
            lm_results['session'].append(session)
            lm_results['block'].append(test_data[session]['block_num'][trial])
            lm_results['trial'].append(test_data[session]['trial_num'][trial])
            lm_results['true_sentence'].append(test_data[session]['sentence_label'][trial] if EVAL_SPLIT == 'val' else None)
            lm_results['pred_sentence'].append(best_sentence)
            pbar.update(1)
print('Language model decoding complete.')

## (Optional) compute validation WER
Skip this section when running on the test split because ground-truth text is not available.

In [ ]:
if EVAL_SPLIT == 'val':
    total_true_length = 0
    total_edit_distance = 0
    for i in range(len(lm_results['pred_sentence'])):
        true_sentence = remove_punctuation(lm_results['true_sentence'][i]).strip()
        pred_sentence = remove_punctuation(lm_results['pred_sentence'][i]).strip()
        ed = editdistance.eval(true_sentence.split(), pred_sentence.split())
        total_true_length += len(true_sentence.split())
        total_edit_distance += ed
        print(f"{lm_results['session'][i]} - Block {lm_results['block'][i]}, Trial {lm_results['trial'][i]}")
        print(f'True sentence:      {true_sentence}')
        print(f'Predicted sentence: {pred_sentence}')
        print(f'WER: {ed} / {len(true_sentence.split())} = {ed / len(true_sentence.split()):.2%}')
        print()
    print(f'Aggregate WER: {100 * total_edit_distance / total_true_length:.2f}%')

## Write submission file
Kaggle expects a two-column CSV with `id` and `text`.

In [ ]:
ids = list(range(len(lm_results['pred_sentence'])))
df_out = pd.DataFrame({'id': ids, 'text': lm_results['pred_sentence']})
df_out.to_csv(SUBMISSION_PATH, index=False)
print('Saved submission to', SUBMISSION_PATH)

In [ ]:
# Stop the language model process if it is still running.
if 'lm_process' in globals() and lm_process.poll() is None:
    lm_process.terminate()
    try:
        lm_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        lm_process.kill()
print('Cleaned up LM process.')